# 02 PD Modeling

Phase 2 builds a baseline probability of default model using the Phase 1 modeling sample. This notebook does not build IFRS 9 staging, Expected Credit Loss calculations, or dashboard logic.

## Project Context

The Credit Risk and IFRS 9 Expected Credit Loss Engine is a finance and risk analytics portfolio project. Phase 1 prepared `data/processed/modeling_sample.csv`; Phase 2 uses that file to train and evaluate a simple, interpretable PD model.

## Modeling Objective

Train a baseline logistic regression classifier that estimates the probability that a loan record belongs to the default class. The output `pd_score` will be used as an input candidate for the Phase 3 ECL engine.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import FIGURES_DIR, MODEL_DIR, OUTPUTS_DIR, PROCESSED_DATA_DIR
from src.features import build_feature_matrix, clean_feature_values, identify_feature_types
from src.modeling import (
    build_logistic_regression_pipeline,
    evaluate_binary_classifier,
    generate_pd_scores,
    save_model,
    split_train_test,
    train_model,
)
from src.visualization import (
    plot_confusion_matrix,
    plot_default_rate_by_score_band,
    plot_pd_score_distribution,
    plot_roc_curve,
    plot_top_coefficients,
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUTS_DIR / "predictions").mkdir(parents=True, exist_ok=True)

## Load Modeling Sample

In [ ]:
modeling_sample_path = PROCESSED_DATA_DIR / "modeling_sample.csv"
if not modeling_sample_path.exists():
    raise FileNotFoundError(f"Missing modeling sample: {modeling_sample_path}")

df = pd.read_csv(modeling_sample_path, low_memory=False)
df = clean_feature_values(df)
print(f"Loaded modeling sample: {modeling_sample_path}")
print(f"Shape: {df.shape}")
display(df.head())

## Target Review

In [ ]:
target_col = "default_flag"
if target_col not in df.columns:
    raise KeyError("default_flag is required for PD modeling.")

target_review = df[target_col].value_counts().sort_index().to_frame("row_count")
target_review["share"] = df[target_col].value_counts(normalize=True).sort_index().round(4)
display(target_review)
print(f"Observed default rate: {df[target_col].mean():.2%}")

## Feature Selection

Phase 2 uses the available candidate features created in Phase 1. `loan_status` is not used as a feature because it directly defines the target.

In [ ]:
X, y = build_feature_matrix(df, target_col=target_col)
numeric_features, categorical_features = identify_feature_types(pd.concat([X, y], axis=1), target_col=target_col)

print(f"Feature matrix shape: {X.shape}")
print(f"Numeric features ({len(numeric_features)}): {numeric_features}")
print(f"Categorical features ({len(categorical_features)}): {categorical_features}")
display(pd.DataFrame({"feature": X.columns, "dtype": [str(dtype) for dtype in X.dtypes]}))

## Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = split_train_test(X, y, test_size=0.2, random_state=42)
print(f"Training rows: {len(X_train):,}")
print(f"Test rows: {len(X_test):,}")
print(f"Train default rate: {y_train.mean():.2%}")
print(f"Test default rate: {y_test.mean():.2%}")

## Preprocessing Pipeline

Numeric features use median imputation and scaling. Categorical features use most-frequent imputation and one-hot encoding with unknown categories ignored.

In [ ]:
model = build_logistic_regression_pipeline(numeric_features, categorical_features)
model

## Baseline Logistic Regression Model

In [ ]:
model = train_model(model, X_train, y_train)
model_path = save_model(model, MODEL_DIR / "pd_logistic_regression.joblib")
print(f"Saved model artifact: {model_path}")

## Model Evaluation

In [ ]:
metrics = evaluate_binary_classifier(model, X_test, y_test)
metric_table = pd.DataFrame(
    {
        "metric": ["roc_auc", "accuracy", "precision", "recall", "f1"],
        "value": [metrics["roc_auc"], metrics["accuracy"], metrics["precision"], metrics["recall"], metrics["f1"]],
    }
)
display(metric_table)
print(metrics["classification_report"])
display(pd.DataFrame(metrics["confusion_matrix"], index=["actual_0", "actual_1"], columns=["pred_0", "pred_1"]))

image = plot_roc_curve(metrics["fpr"], metrics["tpr"], metrics["roc_auc"])
image.save(FIGURES_DIR / "pd_roc_curve.png")
display(image)

image = plot_confusion_matrix(metrics["confusion_matrix"])
image.save(FIGURES_DIR / "pd_confusion_matrix.png")
display(image)

## PD Score Generation

In [ ]:
pd_scores = generate_pd_scores(model, X)
predictions = df.copy()
predictions["row_id"] = predictions.index
predictions["pd_score"] = pd_scores

score_band_labels = ["Very Low", "Low", "Medium", "High", "Very High"]
try:
    predictions["pd_score_band"] = pd.qcut(predictions["pd_score"], q=5, labels=score_band_labels)
    banding_method = "quantile"
except ValueError:
    ranked_scores = predictions["pd_score"].rank(method="first")
    predictions["pd_score_band"] = pd.qcut(ranked_scores, q=5, labels=score_band_labels)
    banding_method = "rank-based fallback"

print(f"PD score banding method: {banding_method}")
display(predictions[["row_id", "default_flag", "pd_score", "pd_score_band"]].head())

image = plot_pd_score_distribution(predictions["pd_score"])
image.save(FIGURES_DIR / "pd_score_distribution.png")
display(image)

band_summary = predictions.groupby("pd_score_band", observed=False).agg(
    row_count=("default_flag", "size"),
    observed_default_rate=("default_flag", "mean"),
    min_pd_score=("pd_score", "min"),
    max_pd_score=("pd_score", "max"),
)
band_summary["observed_default_rate"] = band_summary["observed_default_rate"].round(4)
display(band_summary)

image = plot_default_rate_by_score_band(predictions, band_col="pd_score_band", target_col="default_flag")
image.save(FIGURES_DIR / "pd_default_rate_by_score_band.png")
display(image)

## Feature Importance or Coefficient Review

In [ ]:
coefficients = pd.DataFrame()
try:
    feature_names = model.named_steps["preprocessor"].get_feature_names_out()
    coefficient_values = model.named_steps["classifier"].coef_[0]
    coefficients = pd.DataFrame({"feature": feature_names, "coefficient": coefficient_values})
    coefficients["abs_coefficient"] = coefficients["coefficient"].abs()
    display(coefficients.sort_values("abs_coefficient", ascending=False).head(25))

    image = plot_top_coefficients(coefficients, top_n=20)
    image.save(FIGURES_DIR / "pd_top_coefficients.png")
    display(image)
except Exception as exc:
    print(f"Coefficient extraction skipped: {exc}")

## Save PD Predictions

In [ ]:
requested_original_fields = [
    "loan_amnt",
    "funded_amnt",
    "term",
    "int_rate",
    "installment",
    "grade",
    "sub_grade",
    "emp_length",
    "home_ownership",
    "annual_inc",
    "verification_status",
    "purpose",
    "dti",
    "delinq_2yrs",
    "revol_util",
    "total_acc",
    "loan_status",
]
available_original_fields = [column for column in requested_original_fields if column in predictions.columns]
missing_original_fields = [column for column in requested_original_fields if column not in predictions.columns]

prediction_columns = ["row_id", "default_flag", "pd_score", "pd_score_band"] + available_original_fields
pd_predictions = predictions[prediction_columns].copy()
prediction_path = OUTPUTS_DIR / "predictions" / "pd_predictions.csv"
pd_predictions.to_csv(prediction_path, index=False)

print(f"Saved PD predictions: {prediction_path}")
print(f"Prediction file shape: {pd_predictions.shape}")
print(f"Missing requested original fields not present in modeling sample: {missing_original_fields}")
display(pd_predictions.head())

## Business Interpretation

In [ ]:
print(f"Baseline ROC AUC: {metrics['roc_auc']:.3f}")
print(f"Baseline recall: {metrics['recall']:.3f}")
print(f"Baseline precision: {metrics['precision']:.3f}")
print("The model is a benchmark PD scoring model, not a production credit decision system.")
print("PD score bands provide an ordinal risk segmentation that can feed Phase 3 ECL scenario design.")

## Model Limitations

- This is a baseline logistic regression model only.
- The model is trained on the Phase 1 50,000-row modeling sample, not the full LendingClub file.
- Current accounts are mapped as non-default based on the Phase 1 target definition; this assumption should be reviewed before interpreting lifetime risk.
- No reject inference, macroeconomic adjustment, IFRS 9 staging, LGD, EAD, or ECL calculation is included in Phase 2.
- Coefficients are useful for directional review but require care because categorical variables are one-hot encoded and numeric variables are scaled.

## Outputs for Phase 3 ECL Engine

- `outputs/model/pd_logistic_regression.joblib`
- `outputs/predictions/pd_predictions.csv`
- Phase 2 evaluation figures in `reports/figures/`

These outputs are candidate inputs for Phase 3. Phase 3 still needs explicit LGD, EAD, and ECL assumptions.

## Next Steps

- Confirm target policy assumptions with special attention to current and early delinquency accounts.
- Use the PD prediction file as the starting point for Phase 3 ECL design.
- Define transparent LGD and EAD assumptions.
- Build the ECL calculation notebook without adding IFRS 9 staging complexity until the basic engine is correct.